### Sistema Recomendación Híbrido

El objetivo de este proyecto es construir un sistema de recomendación basado en las compras realizadas por los clientes de una superficie comercial. El enfoque parte de una restricción clave: al tratarse de compra física, no es posible recomendar en tiempo real durante la compra, ya que no conocemos la cesta del cliente hasta que pasa por caja. Esto diferencia el problema del de una tienda online, donde sí se puede recomendar según la cesta se va componiendo.
Por ello, el sistema se orienta a la recomendación entre visitas: a partir del historial de compra de cada cliente, generar recomendaciones personalizadas que incentiven su siguiente visita (mediante email, app o cupones en el ticket), en línea con los programas de fidelización habituales en el sector.
Para ello se construyen tres modelos:

- ALS, que aprende los hábitos de cada cliente a partir de su historial para predecir y recomendar su próxima compra. Es el núcleo de la personalización.
- Apriori, que analiza la composición de las cestas para extraer patrones de co-compra. Más que recomendar al cliente directamente, alimenta decisiones de negocio: colocación de productos en tienda, promociones cruzadas y cupones post-compra.
- Popularidad, que recomienda los productos más vendidos como red de seguridad para clientes nuevos o sin historial suficiente.

### Librerías

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [103]:
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
import optuna
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [3]:
df = pd.read_pickle('data/transacciones_limpio.pkl')

df.head(5)

,line_id,ticket_number,user_card_id,payment_method,shop_name,shop_address,product_code,product_name,seccion,units,price_per_unit,price_total,created_at
0,9550698,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1041,HELADO VAINILLA 1L,Congelados,1,3.41,3.41,2025-01-01 08:20:00
1,9550697,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1008,AZUCAR BLANCO 1KG,Alimentacion seca,1,1.06,1.06,2025-01-01 08:20:00
2,9550707,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1055,BOLSAS BASURA 30L,Drogueria,1,1.98,1.98,2025-01-01 08:20:00
3,9550706,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1022,CARNE PICADA MIXTA 500G,Frescos,1,3.95,3.95,2025-01-01 08:20:00
4,9550693,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1068,ARENA GATO 5L,Mascotas,2,3.94,7.88,2025-01-01 08:20:00


Comprobamos longitud datos y fecha mínima y máxima para realizar división de datos

In [4]:
print(f'Filas: {len(df):,}')
print(df.created_at.min())
print(df.created_at.max())

Filas: 50,000
2025-01-01 08:20:00
2025-06-30 21:21:00


Tras comprobar fechas procedemos a separar los datos y crear un conjunto de entrenamiento y testeo. Esta separación se realiza de forma manual.

In [5]:
# Fecha corte. Últimos 30 días para test
fecha_corte = df.created_at.max() - pd.Timedelta(days=30)
print(f'Fecha corte: {fecha_corte}')

# creamos conjuntos
train = df[df.created_at <= fecha_corte].copy()
test = df[df.created_at > fecha_corte].copy()

print(f'Longitud Train: {len(train)}; ({len(train)/len(df)*100:.0f}%)') 
print(f'Longitud Test: {len(test)}; ({len(test)/len(df)*100:.0f}%)')
print('-'*50)
print(f'Tickets train: {train.ticket_number.nunique():,}') # comprobación de tickets por conjunto
print(f'Tickets test: {test.ticket_number.nunique():,}')


Fecha corte: 2025-05-31 21:21:00
Longitud Train: 41649; (83%)
Longitud Test: 8351; (17%)
--------------------------------------------------
Tickets train: 4,340
Tickets test: 856


Ahora vamos a comprobar el número de clientes evaluables. Este paso es importante porque determina la frontera a partir de la cual entra en juego cada modelo. Para ALS, un sistema personalizado, se necesita un registro histórico del cliente por tanto no funcionará con aquellos clientes nuevos, lo que se conoce como cold start. Estos clientes de cold start serán asignados a sistemas como Apriori o popularidad para realizar la recomendación.

In [6]:
# almacenamos los clientes en conjuntos y eliminamos duplicados
clientes_train = set(train.user_card_id.unique())
clientes_test = set(test.user_card_id.unique())

evaluables = clientes_test & clientes_train # intersección que guarda solo clientes que se encuentran en ambos conjuntos. Aparecen tanto en train como test
cold_start = clientes_test - clientes_train  # clientes que compraron por primera vez dentro del rango de test.

print(f'Clientes en test: {len(clientes_test)}')
print(f'Evaluables: {len(evaluables)}') # clientes sobre los que se puede probar ALS
print(f'Cold start puro (test): {len(cold_start)}')

Clientes en test: 583
Evaluables: 537
Cold start puro (test): 46


Con estos cálculos ya tenemos una idea sobre la cantidad de clientes que vamos a manejar en las evaluaciones de modelos.
Contamos con 583 clientes dentro del conjunto de testeo de los cuales, 46 de ellos han realizado su primera compra en el periodo que contempla el conjunto de test. Los restantes 537 clientes se encuentran en ambos conjuntos y nos ayudarán a medir si ALS acierta. 

### Modelo Basado en Popularidad

El primer modelo que se construye es un sistema NO personalizado. Este modelo solo se va a fijar en cuantas veces aparece cada producto en cada ticket. Se trata de un modelo totalmente sencillo que no necesita conocer ningún cliente, es ideal para cuando estamos empezando o el cliente es nuevo, se usa como referencia base.

In [7]:
# Popularidad de producto según tickets en conjunto train
popularidad = (
    train.groupby('product_name')['ticket_number']
    .nunique()
    .sort_values(ascending=False)
)

print('Top 10 productos más populares (por nº de tickets):')
print(popularidad.head(10))

Top 10 productos más populares (por nº de tickets):
product_name
CHAMPU ANTICASPA 400ML      3393
LECHUGA ICEBERG UD          2843
YOGUR NATURAL PACK 4        2231
HELADO VAINILLA 1L          1855
ARROZ REDONDO 1KG           1643
TOMATE RAMA KG              1610
ENSALADA CESAR PREPARADA    1336
CERVEZA PACK 6              1264
SAL FINA 1KG                1117
ARENA GATO 5L               1073
Name: ticket_number, dtype: int64


In [8]:
def recomendar_popularidad(n=10):
    """
    Función para recomendar productos en base a la popularidad. Veces que aparece en tickets
    Parámetros:
    -n: número de productos seleccionados
    """
    return popularidad.head(n).index.tolist()

print(recomendar_popularidad(10))

['CHAMPU ANTICASPA 400ML', 'LECHUGA ICEBERG UD', 'YOGUR NATURAL PACK 4', 'HELADO VAINILLA 1L', 'ARROZ REDONDO 1KG', 'TOMATE RAMA KG', 'ENSALADA CESAR PREPARADA', 'CERVEZA PACK 6', 'SAL FINA 1KG', 'ARENA GATO 5L']


Vamos a evaluar el comportamiento de este sistema base

In [9]:
recomendados = set(popularidad.head(10).index) # creamos un conjunto con los 10 productos más populares

# Comprobamos cliente de forma individual
cliente = list(evaluables)[0] # creamos una lista con los clientes y seleccionamos el primer cliente
comprados = set(test[test['user_card_id'] == cliente]['product_name']) # conjunto de productos comprados por el cliente dentro de test

print(f'Cliente: {cliente}')
print(f'Productos comprados en conjunto test: {len(comprados)}')
print(comprados)

Cliente: 100352
Productos comprados en conjunto test: 23
{'ARROZ REDONDO 1KG', 'LECHE ENTERA BRIK 1L', 'PIZZA CUATRO QUESOS', 'MERLUZA FILETE KG', 'TOMATE FRITO 400G', 'CAFE MOLIDO 250G', 'REFRESCO COLA 2L', 'HUEVOS DOCENA M', 'AZUCAR BLANCO 1KG', 'MACARRONES 500G', 'HELADO VAINILLA 1L', 'TOMATE RAMA KG', 'ARENA GATO 5L', 'SAL FINA 1KG', 'PECHUGA POLLO BANDEJA', 'MANZANA GOLDEN KG', 'LECHUGA ICEBERG UD', 'PAPILLA CEREALES 600G', 'CHAMPU ANTICASPA 400ML', 'LECHE INFANTIL CONTINUACION', 'ENSALADA CESAR PREPARADA', 'PATATA KG', 'YOGUR NATURAL PACK 4'}


Guardamos en una variable los productos comprados por un cliente concreto durante el periodo de testeo. Ahora vamos a comparar este cliente con el top de productos más populares y ver la tasa de acierto.

In [10]:
aciertos = recomendados & comprados # creamos una intersección entre productos recomendados y comprados del cliente seleccionado

print(f'Nº de aciertos: {len(aciertos)}')
print(f'Aciertos: {aciertos}')

Nº de aciertos: 9
Aciertos: {'ARROZ REDONDO 1KG', 'CHAMPU ANTICASPA 400ML', 'TOMATE RAMA KG', 'ARENA GATO 5L', 'ENSALADA CESAR PREPARADA', 'SAL FINA 1KG', 'YOGUR NATURAL PACK 4', 'LECHUGA ICEBERG UD', 'HELADO VAINILLA 1L'}


De los 23 productos comprados por el cliente en el conjunto de test, 9 de ellos se encuentran dentro de los 10 más populares

In [11]:
precision = len(aciertos) / 10 # cuantos valores acertamos
recall = len(aciertos) / len(comprados) # de los productos comprados, cuantos cubrimos

print(f'Precision@10 para el cliente: {precision:.2f}')
print(f'Recall@10 para el cliente: {recall:.2f}')

Precision@10 para el cliente: 0.90
Recall@10 para el cliente: 0.39


La precision alta es esperable en un baseline de popularidad cuando los clientes consumen mayoritariamente productos populares. Por el contrario, el recall queda limitado de forma estructural, al recomendar solo 10 productos de 23 que compra este cliente, el valor máximo que se puede alcanzar es 0.43, siendo este resultado una métrica comparativa entre modelos.

In [12]:
# inicializamos lista para almacenar resultados de todos los clientes
precisiones = []
recalls = []

# inicializamos bucle
for c in evaluables:
    productos_comprados = set(test[test['user_card_id'] == c]['product_name'])

    productos_acertados = len(recomendados & productos_comprados) # intersección entre productos comprados y top 10 más populares.

    precisiones.append(productos_acertados/10) # añadimos la precision de cada cliente al listado
    recalls.append(productos_acertados/ len(productos_comprados)) # añadimos el recall de cada cliente al listado.

print(f'Cliente evaluados: {len(precisiones)}')
print(f'Precision@10 media : {np.mean(precisiones):.4f}')
print(f'Recall@10 media : {np.mean(recalls):.4f}')


Cliente evaluados: 537
Precision@10 media : 0.5058
Recall@10 media : 0.4515


Tras realizar el mecanismo de forma individual para un solo cliente, replicamos lo mismo en bucle para todos los clientes que aparecen en train y test. Esto devuelve una precision media de 0.5058, es decir, 5 de cada 10 productos recomendados acaban siendo compras reales del cliente. Se puede decir que es un buen resultado para un modelo NO personalizado.

En cuanto al Recall, este indica que los productos acertados representan el 45% de lo que el cliente compró.

El rendimiento del modelo no personalizado obtiene unos resultados muy buenos que pueden ser difíciles de superar para un modelo basado en ALS. El principal motivo de estos resultados es el propio dataset, no existe una gran diferenciación de clientes y el número de productos que se maneja es pequeño en comparación a un entorno real. Aunque se trate de datos sintéticos, esto no implica que en un entorno real, siempre vaya a tener mejor resultado un modelo más complejo como ALS, para una superficie tipo supermercado, recomendar simplemente los productos más vendidos puede ser más eficaz que tratar de construir un modelo más complejo.

In [13]:
pop_train = set(popularidad.head(10).index)
pop_test = set(test.groupby('product_name')['ticket_number'].nunique()
               .sort_values(ascending=False).head(10).index)

print(f'Productos en común entre top-10 train y top-10 test: {len(pop_train & pop_test)}/10')

Productos en común entre top-10 train y top-10 test: 9/10


Por último, comprobamos como de estable es el top 10 de productos entre train y test. El resultado que se obtiene es que 9 de cada 10 productos coinciden entre ambos conjuntos. (SUBIR ESTO ARRIBA TIENE MÁS SENTIDO)

### Sistema ALS

Mientras que el modelo basado en popularidad es un sistema no personalizado, el sistema ALS es un sistema de recomendación personalizado, en donde cada cliente recibe una recomendación acorde a su perfil.

El método Alternating Least Squares (ALS) consiste en crear una matriz cliente-producto en donde se registran las veces que cada cliente ha comprado cada producto. Esa matriz será la capa de entrada al modelo y donde empieza la factorización, partiendo de esta matriz ALS la descompone en dos matrices más pequeñas, una de clientes y otra de productos, descritas por factores latentes. Los factores latentes son características ocultas que el modelo inventa para explicar los datos. Este factor latente es similar a un ajuste de parámetros en ML clásico. 

Con esto, ALS congela una matriz mientras resuelve la otra y una vez tiene una resuelta, repite el proceso para la que aún no está resuelta apoyandose en la otra matriz. Una vez se ha completado este proceso, se realizan las prediciones que consiste en multiplicar los vecoters de factores. Si estos vectores están alineados, la predicción es alta y se devuelve la predicción más alta como recomendación.

La idea de seleccionar este método frente a otros se encuentra en que en un entorno real, la matriz de cliente-producto es muy dispersa y por tanto un sistema item-item tendrá complicaciones para realizar las recomendaciones. En este ejercicio la matriz con la que se trabaja no es dispersa.

In [ ]:
# Frecuencia, nº de veces que cada cliente compró cada producto
interacciones = (train.groupby(['user_card_id', 'product_name'])
                 .size()
                 .reset_index(name='frecuencia'))

print(f'Interaciones cliente-producto: {len(interacciones)}')
interacciones.head()

Interaciones cliente-producto: 26204


,user_card_id,product_name,frecuencia
0,100001,CARNE PICADA MIXTA 500G,1
1,100001,CEBOLLA KG,1
2,100001,CHAMPU ANTICASPA 400ML,2
3,100001,ENSALADA CESAR PREPARADA,1
4,100001,HELADO VAINILLA 1L,1


Como la librería **implicit** trabaja con matrices donde filas y columnas son índices enteros hay que aplicar un mapeo.

In [15]:
# lista única productos y clientes
clientes_unicos = interacciones['user_card_id'].unique()
productos_unicos = interacciones['product_name'].unique()

# Mapeo
cliente_idx = {c: i for i, c in enumerate(clientes_unicos)} # compresión diccionario
productos_idx = {p: i for i, p in enumerate(productos_unicos)} # compresión diccionario

# Mapeo inverso; interpretación de resultados posterior
idx_cliente = {i: c for c, i in cliente_idx.items()}
idx_producto = {i: p for p, i in productos_idx.items()}

print(f'Clientes: {len(clientes_unicos)}')
print(f'Productos: {len(productos_unicos)}')


Clientes: 1296
Productos: 68


Una vez mapeados clientes y productos es hora de construir la matriz que recibe ALS. La matriz dispersa solo almacena las celdas que contienen valores ignorando aquellas que son cero. Para estos datos, tal vez no se aprecia su efecto, pero para grandes superficies supone una diferencia grande en cuanto a tiempo.

In [16]:
# Asignamos valores mapeados a cada valor.
filas = interacciones['user_card_id'].map(cliente_idx)
columnas = interacciones['product_name'].map(productos_idx)
valores = interacciones['frecuencia'].astype(float)

# Matriz dispersa cliente x producto
matriz_cp = csr_matrix(
    (valores, (filas,columnas)),
    shape=(len(clientes_unicos), len(productos_unicos))
)

print(f'Forma matriz: {matriz_cp.shape}')
print(f'Valores NO nulos: {matriz_cp.nnz:}')
print(f'Densidad: {matriz_cp.nnz / (matriz_cp.shape[0]*matriz_cp.shape[1])*100:.2f}%')

Forma matriz: (1296, 68)
Valores NO nulos: 26204
Densidad: 29.73%


Es importante tener en cuenta que se construye la matriz sobre el conjunto TRAIN por tanto el número de clientes se reduce ya que no todos los clientes de TRAIN se encuentran en TEST. Este método solo puede aprender con aquellos clientes que tienen historial, por tanto los nuevos clientes, los que se clasifican como cold start quedan fuera de este modelo.

Con la matriz creada vamos a iterar con ALS por primera vez, los valores que van a recibir los parámetros son los propios valores predeterminados que vienen con el modelo, sin ningún tipo de ajuste. Posteriormente se ajustarán estos parámetros mediante Optuna.

In [17]:
# modelo base
modelo_base_als = AlternatingLeastSquares(random_state=42)

modelo_base_als.fit(matriz_cp)
print('Modelo base entrenado')

  0%|          | 0/15 [00:00<?, ?it/s]

Modelo base entrenado


In [44]:
cliente_real = clientes_unicos[0] # ID real del cliente
cliente_index = cliente_idx[cliente_real] # indice en matriz

# Generación de recomendaciones
ids, scores = modelo_base_als.recommend(
    cliente_index,
    matriz_cp[cliente_index], # fila del cliente en matriz
    N=10, # top 10
    filter_already_liked_items=True # no recomendamos productos ya comprados
)

# Pasamos los índices de vuelta a nombres de producto
recomendaciones_als = [idx_producto[i] for i in ids]

print(f'Cliente: {cliente_real}')
print('Recomendaciones')
for prod, score in zip(recomendaciones_als, scores):
    print(f'{prod:35s} {score:.3f}')

Cliente: 100001
Recomendaciones
PIZZA JAMON Y QUESO                 0.012
CERVEZA PACK 6                      0.011
PIZZA CUATRO QUESOS                 0.010
REFRESCO NARANJA 2L                 0.009
REFRESCO COLA 2L                    0.009
LASAÑA BOLOÑESA 400G                0.008
ARENA GATO 5L                       0.008
VINO TINTO CRIANZA                  0.008
ARROZ REDONDO 1KG                   0.006
GUISANTES CONGELADOS 1KG            0.006


In [45]:
print("\nLo que compró en train:")
set(train[train["user_card_id"] == cliente_real]["product_name"])


Lo que compró en train:


{'CARNE PICADA MIXTA 500G',
 'CEBOLLA KG',
 'CHAMPU ANTICASPA 400ML',
 'ENSALADA CESAR PREPARADA',
 'HELADO VAINILLA 1L',
 'LECHUGA ICEBERG UD',
 'MANTEQUILLA 250G',
 'MERLUZA FILETE KG',
 'PAN BARRA RUSTICA',
 'PASTA ESPAGUETI 500G',
 'PATATA KG',
 'QUESO LONCHAS 200G',
 'SAL FINA 1KG',
 'SUAVIZANTE CONCENTRADO 1.5L',
 'TOMATE FRITO 400G',
 'TOMATE RAMA KG',
 'TORTILLA PATATA REFRIGERADA',
 'YOGUR NATURAL PACK 4'}

Observando las recomendaciones y puntuaciones que devuelve el modelo base con los productos que ha comprado en el conjunto train, se puede ver como el modelo base apenas está discriminando. Este es un resultado esperable ya que el modelo se encuentra sin ajustar y por tanto no puede encontrar factores latentes que hagan diferenciación de productos.

Ahora toca realizar la evaluación de resultados para este único cliente.

In [46]:
comprados_als_test = set(test[test['user_card_id'] == cliente_real]['product_name'])
print(f'Comprado en test ({len(comprados_als_test)} productos):')
print(comprados_als_test)

Comprado en test (5 productos):
{'CHAMPU ANTICASPA 400ML', 'TOMATE RAMA KG', 'VERDURA WOK CONGELADA 750G', 'ZUMO NARANJA 1L', 'SUAVIZANTE CONCENTRADO 1.5L'}


In [47]:
aciertos_als = set(recomendaciones_als) & comprados_als_test
precision_als = len(aciertos_als) / 10
recall_als = len(aciertos_als) / len(comprados_als_test) if len(comprados_als_test) > 0 else 0

print(f'Aciertos: {aciertos_als}')
print(f'Precision@10: {precision_als:.2f}')
print(f'Recall@10:    {recall_als:.2f}')

Aciertos: set()
Precision@10: 0.00
Recall@10:    0.00


In [48]:
print("RECOMENDADO por ALS:")
print(set(recomendaciones_als))
print()
print("COMPRADO en test:")
print(comprados_als_test)
print()
print("¿El cliente está en test?", cliente_real in test["user_card_id"].values)
print("Nº productos comprados en test:", len(comprados_als_test))

RECOMENDADO por ALS:
{'ARROZ REDONDO 1KG', 'CERVEZA PACK 6', 'LASAÑA BOLOÑESA 400G', 'ARENA GATO 5L', 'PIZZA CUATRO QUESOS', 'VINO TINTO CRIANZA', 'PIZZA JAMON Y QUESO', 'REFRESCO COLA 2L', 'GUISANTES CONGELADOS 1KG', 'REFRESCO NARANJA 2L'}

COMPRADO en test:
{'CHAMPU ANTICASPA 400ML', 'TOMATE RAMA KG', 'VERDURA WOK CONGELADA 750G', 'ZUMO NARANJA 1L', 'SUAVIZANTE CONCENTRADO 1.5L'}

¿El cliente está en test? True
Nº productos comprados en test: 5


Teniendo en cuenta que se está evaluando con filtro activado, es decir, que no se recomienden productos ya comprados, para el cliente unitario donde se comprueba el sistema, el modelo base tiene nula capacidad de recomendación. Antes de pasar a un ajuste de modelo, se va a comprobar con todo el conjunto de clientes para ver los valores medios de evaluación.

In [ ]:
# # inicializamos listas
# precisiones_als = []
# recalls_als = []

# for cliente in evaluables:
#     cliente_index = cliente_idx[cliente]

#     ids, scores = modelo_base_als.recommend(
#         cliente_index,
#         matriz_cp[cliente_index], # fila del cliente en matriz
#         N=10, # top 10
#         filter_already_liked_items=True # no recomendamos productos ya comprados
#     )

#     recomendados_als = set(idx_producto[i] for i in ids)
#     comprados_als = set(test[test['user_card_id'] == cliente]['product_name'])

#     if len(comprados_als) == 0:
#         continue

#     aciertos_als = len(recomendados_als & comprados_als)
#     precisiones_als.append(aciertos_als/10)
#     recalls_als.append(aciertos_als/len(comprados_als))

# print(f'Clientes evaluados: {len(precisiones_als)}')
# print(f'Precision@10: {np.mean(precisiones_als):.2f}')
# print(f'Recall@10:    {np.mean(recalls_als):.2f}')
    

Es importante tener en cuenta que sistema ALS para un tamaño de datos pequeño rara vez logrará batir a un sistema de recomendación. De hecho, en este momento, la comparación no es justa ya que el sistema ALS está excluyendo aquellos productos que ya han sido comprados por el cliente, es decir, busca en este momento descrubir productos nuevos que recomendar, una tarea que resulta complicada de por si. Por tanto el resultado a batir no son los resultados del sistema de popularidad, sino que se busca batir estos números del modelo base con el modelo optimizado.

Para evaluar con Optuna, vamos a crear una función con el código que ya hemos empleado en varias ocasiones.

In [84]:
def evaluar_als(modelo, matriz, clientes, k=10, filtrar=True):
    precisiones_als = []
    recalls_als = []

    for cliente in clientes:
        cliente_index = cliente_idx[cliente]

        ids, _ = modelo.recommend(
            cliente_index,
            matriz[cliente_index], # fila del cliente en matriz
            N=k, # top 10
            filter_already_liked_items=filtrar # no recomendamos productos ya comprados
        )

        recomendados_als = set(idx_producto[i] for i in ids)
        comprados_als = set(test[test['user_card_id'] == cliente]['product_name'])

        if len(comprados_als) == 0:
            continue

        aciertos_als = len(recomendados_als & comprados_als)
        precisiones_als.append(aciertos_als/k)
        recalls_als.append(aciertos_als/len(comprados_als))

    return np.mean(precisiones_als), np.mean(recalls_als)

La función nos devuelve la media de precision y recall calculada sobre cada cliente que es evaluable.

In [87]:
prec, rec = evaluar_als(modelo_base_als, matriz_cp, evaluables, k=10, filtrar=True)
print(f"Precision@10: {prec:.4f}  Recall@10: {rec:.4f}")

Precision@10: 0.1436  Recall@10: 0.1113


Con la función objetivo, se procede a realizar la búsqueda de los mejores parámetros para el modelo ALS, en este caso factores latentes.

- factors: es la opción más importante. Esto va indicar el número de dimensiones ocultas que el modelo va a usar para describir a clientes y productos. Cuantos más factores, mayor capacidad para capturar patrones complejos, pero nos arriesgamos a un sobreajuste
- regularization: se encarga de controlar el sobreajuste penalizando que los factores tomen valores demasiado grandes. Mismo concepto que Ridge, un valor alto fuerza al modelo a ser más conservador y un valor bajo da más libertad.
- alpha: es el factor de confianza y es específio en el feedback implícito. Al trabajar con frecuencias de compra, alpha se encarga de indicar cuanta confianza hay en esas frecuencias.
- iterations: indica el número de veces que hace el cálculo de cada matriz, es decir, el número de veces que refinamos el algoritmo.

In [ ]:
def objetivo(trial):
    # Espacio de búsqueda
    factors = trial.suggest_int('factors', 5,100) # factores latentes
    regularization = trial.suggest_float('regularization', 0.001, 1.0, log=True) # regularización de sobreajuste
    alpha = trial.suggest_float('alpha', 1.0, 4.0) # confianza para feedback implícito
    iterations = trial.suggest_int('iterations', 10, 30) # iteracciones en matrices

    # Entrenar ALS
    modelo = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        alpha=alpha,
        iterations=iterations,
        random_state=42 # semilla para replicar resultados.
    )
    modelo.fit(matriz_cp)

    # Evaluación con filtro activado
    precision, _ = evaluar_als(modelo, matriz_cp, evaluables, k=10, filtrar=True)
    return precision

Una vez tenemos la función creada empezamos con el estudio.

In [75]:
study = optuna.create_study(direction='maximize')
study.optimize(objetivo, n_trials=30, show_progress_bar=False)

print('Mejor precision: ', study.best_value)
print('Mejores parámetros: ', study.best_params)

[I 2026-06-30 12:26:18,703] A new study created in memory with name: no-name-90185a25-5a3a-4f3e-8b4a-71ad99faf1a5


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-06-30 12:26:18,964] Trial 0 finished with value: 0.13966480446927373 and parameters: {'factors': 38, 'regularization': 0.06294147502515794, 'alpha': 1.7553098248181598, 'iterations': 12}. Best is trial 0 with value: 0.13966480446927373.


  0%|          | 0/19 [00:00<?, ?it/s]

[I 2026-06-30 12:26:19,273] Trial 1 finished with value: 0.17486033519553074 and parameters: {'factors': 25, 'regularization': 0.1974438096815714, 'alpha': 2.829910693631739, 'iterations': 19}. Best is trial 1 with value: 0.17486033519553074.


...

  0%|          | 0/14 [00:00<?, ?it/s]

[I 2026-06-30 12:26:26,842] Trial 28 finished with value: 0.17467411545623837 and parameters: {'factors': 23, 'regularization': 0.017844960295505956, 'alpha': 3.7975661837596526, 'iterations': 14}. Best is trial 12 with value: 0.22197392923649908.


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-06-30 12:26:27,085] Trial 29 finished with value: 0.15716945996275605 and parameters: {'factors': 35, 'regularization': 0.09182572998524394, 'alpha': 3.6180109793882784, 'iterations': 12}. Best is trial 12 with value: 0.22197392923649908.


Mejor precision:  0.22197392923649908
Mejores parámetros:  {'factors': 5, 'regularization': 0.9702711465727375, 'alpha': 1.058479065847884, 'iterations': 12}


Una vez se ha realizado el estudio y encontrado los mejores parámetros dentro de los rangos de búsqueda indicados, se procede a crear el modelo con esos parámetros y a entrenar.

In [77]:
modelo_als_final = AlternatingLeastSquares(**study.best_params, random_state=42)

modelo_als_final.fit(matriz_cp)

prec, rec = evaluar_als(modelo_als_final, matriz_cp, evaluables, k=10, filtrar=True)
print(f"Precision@10: {prec:.4f}  Recall@10: {rec:.4f}")

  0%|          | 0/12 [00:00<?, ?it/s]

Precision@10: 0.2220  Recall@10: 0.1710


In [88]:
# Generamos recomendaciones para unos cuantos clientes y las inspeccionamos
for cliente_real in list(evaluables)[:5]:
    cliente_index = cliente_idx[cliente_real]
    
    ids, scores = modelo_als_final.recommend(
        cliente_index,
        matriz_cp[cliente_index],
        N=5,
        filter_already_liked_items=True
    )
    recomendados = [idx_producto[i] for i in ids]
    
    # Lo que el cliente compra habitualmente (train), para contexto
    compra_habitual = set(train[train["user_card_id"] == cliente_real]["product_name"])
    
    print(f"Cliente {cliente_real}")
    print(f"  Compra habitual: {list(compra_habitual)[:6]}")
    print(f"  Le recomendamos: {recomendados}")


Cliente 100352
  Compra habitual: ['ARROZ REDONDO 1KG', 'TOMATE RAMA KG', 'CHAMPU ANTICASPA 400ML', 'AGUA MINERAL PACK 6', 'ZUMO NARANJA 1L', 'HARINA TRIGO 1KG']
  Le recomendamos: ['HELADO VAINILLA 1L', 'ARENA GATO 5L', 'SAL FINA 1KG', 'CARNE PICADA MIXTA 500G', 'CEBOLLA KG']
Cliente 100355
  Compra habitual: ['ARROZ REDONDO 1KG', 'DESODORANTE SPRAY 200ML', 'LECHE ENTERA BRIK 1L', 'CERVEZA PACK 6', 'TOMATE FRITO 400G', 'CAFE MOLIDO 250G']
  Le recomendamos: ['MERLUZA FILETE KG', 'SALMON FRESCO KG', 'ENSALADA CESAR PREPARADA', 'CARNE PICADA MIXTA 500G', 'PLATANO KG']
Cliente 100356
  Compra habitual: ['ARROZ REDONDO 1KG', 'LECHE ENTERA BRIK 1L', 'TOALLITAS BEBE PACK 3', 'MERLUZA FILETE KG', 'HARINA TRIGO 1KG', 'ZUMO NARANJA 1L']
  Le recomendamos: ['MANTEQUILLA 250G', 'ARENA GATO 5L', 'QUESO LONCHAS 200G', 'LECHE INFANTIL CONTINUACION', 'HUEVOS DOCENA M']
Cliente 100357
  Compra habitual: ['PAÑALES T4 PACK 44', 'ARROZ REDONDO 1KG', 'CERVEZA PACK 6', 'TOALLITAS BEBE PACK 3', 'TOMATE FRI

Para ver el comportamiento del modelo, se generan un par de recomendaciones comparando con la compra habitual del cliente. Los resultados que aporta el modelo, se pueden decir que son un poco lo esperado debido al resultado de las métricas. 

El cliente 100356, entre la compra habitual tiene leche y toallitas de bebe. Una recomendación realizada es leche infantil. Se puede decir que cumple la lógica esperada. 
Sin embargo, también se observa que hay ruido entre los resultados, en el cliente 100352 no hay un patrón de recomendación claro.

Lo más importante a tener en cuenta en esta situación, es que al ser datos sintéticos, no existe una gran diferenciación de clientes. Esto es observable en los resultados de las métricas. Ante la ausencia de diferenciación, en términos generales las recomendaciones no serán muy específicas.

### APRIORI

Como vamos a describir patrones de co-compra y no predecir, vamos a emplear el conjunto de datos completo. La idea de usar apriori es para desarrollar inteligencia de negocio. Que productos combinan entre ellos de forma complementaria buscando aumentar la cesta de la compra mediante cross-selling. Este conocimiento permite desarrollar una mejor distribución de la tienda y crear menos resistencia a los clientes para añadir productos complementarias a la cesta.

In [93]:
# Convertimos cada ticket en una lista de productos
cestas = df.groupby('ticket_number')['product_name'].apply(list)

print(f'Numero de cestas (tickets): {len(cestas)}')
print('Ejemplo de cesta')
print(cestas.iloc[0])

Numero de cestas (tickets): 5196
Ejemplo de cesta
['CHAMPU ANTICASPA 400ML', 'PIZZA JAMON Y QUESO', 'YOGUR NATURAL PACK 4', 'QUESO RALLADO 200G', 'REFRESCO NARANJA 2L', 'ZUMO NARANJA 1L', 'HELADO VAINILLA 1L', 'ROLLO COCINA PACK 4', 'TORTILLA PATATA REFRIGERADA', 'ENSALADA CESAR PREPARADA']


El algoritmo Apriori trabaja con una matriz binaria, por tanto hay que aplicar un encoder que pase los datos a binario. El resultado que vamos a obtener es que cada fila es un ticket y cada columa un producto que recibe un valor True o False

In [97]:
te = TransactionEncoder() #instanciamos objeto
te_array = te.fit_transform(cestas)

# Pasamos array a DF
cestas_binarias = pd.DataFrame(te_array, columns=te.columns_)

print(f'Forma: {cestas_binarias.shape}')
cestas_binarias

Forma: (5196, 68)


,ACEITE OLIVA VIRGEN 1L,AGUA MINERAL PACK 6,ARENA GATO 5L,ARROZ REDONDO 1KG,ATUN CLARO PACK 3,AZUCAR BLANCO 1KG,BOLSAS BASURA 30L,CAFE MOLIDO 250G,CALDO POLLO PACK,CARNE PICADA MIXTA 500G,...,SALMON FRESCO KG,SUAVIZANTE CONCENTRADO 1.5L,TOALLITAS BEBE PACK 3,TOMATE FRITO 400G,TOMATE RAMA KG,TORTILLA PATATA REFRIGERADA,VERDURA WOK CONGELADA 750G,VINO TINTO CRIANZA,YOGUR NATURAL PACK 4,ZUMO NARANJA 1L
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,True
1,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,False,False,True,False,False,False,True,False,False,False,...,True,True,True,True,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5191,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,True,False,False,False,False
5192,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
5193,False,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,False,True,False
5194,False,False,False,True,False,True,False,False,False,True,...,False,False,False,True,True,False,False,False,True,False


Ahora con los datos transformados, es cuando se aplica Apriori para encontrar las combinaciones de productos que aparecen con frecuencia. En esta primera fase empleamos el primer parámetro de decisión, el soporte mínimo, que es el umbral de frecuencia que debe aparecer para cada combinación para ser considerada como una opción

In [102]:
# instanciamos algoritmo e indicamos que la frecuencia de aparición mínima sea un 2%
itemsets = apriori(cestas_binarias, min_support=0.02, use_colnames=True)

itemsets['longitud'] = itemsets['itemsets'].apply(len)

print(f'Itemsets encontados: {len(itemsets)}')
itemsets.sort_values('support', ascending=False).head(10)

Itemsets encontados: 8976


,support,itemsets,longitud
13,0.784065,frozenset({CHAMPU ANTICASPA 400ML}),1
30,0.651848,frozenset({LECHUGA ICEBERG UD}),1
366,0.531370,"frozenset({CHAMPU ANTICASPA 400ML, LECHUGA ICE...",2
66,0.520593,frozenset({YOGUR NATURAL PACK 4}),1
402,0.441109,"frozenset({CHAMPU ANTICASPA 400ML, YOGUR NATUR...",2
23,0.427444,frozenset({HELADO VAINILLA 1L}),1
3,0.375481,frozenset({ARROZ REDONDO 1KG}),1
621,0.373941,"frozenset({LECHUGA ICEBERG UD, YOGUR NATURAL P...",2
62,0.371632,frozenset({TOMATE RAMA KG}),1
359,0.363741,"frozenset({CHAMPU ANTICASPA 400ML, HELADO VAIN...",2


El valor de support que aparece en el DF creado indica la proporción con que aparece ese producto en los tickets. Para el champu anticaspa tenemos que aparece en el 78.4% de los tickets. Cuanto más alto sea ese valor, más común es esa combinación.

La longitud nos indica el número de productos que aparece en cada itemset. 
Para este ejercicio donde buscamos patrones de co-compra, los itemsets de longitud 1 no resultan interesantes, ya que muestran popularidad, no co-compra. Nos interesa encontrar productos complementarios.

Con esto presente, vamos a generar reglas de asocicación.

In [105]:
# regla de asociacion con lift
reglas = association_rules(itemsets, metric='lift', min_threshold=1)

print(f'Reglas generadas: {len(reglas)}')
print(reglas.columns.tolist())

Reglas generadas: 171158
['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski']


Con un soporte mínimo del 2% se crean demasiadas reglas, 171158, lo que resulta completamente inmanejable. El umbral aquí no es una decisión optimizable, es una decisión de negocio. Para poder visualizar esta situación, vamos a dejar este código como base y subir el umbral en otro bloque.

In [107]:
# Reglas accionables: combinaciones frecuentes Y con asociación fuerte
reglas_filtradas = reglas[
    (reglas["support"] >= 0.05) &      # aparece en al menos 5% de cestas
    (reglas["lift"] >= 1.2) &          # asociación notablemente sobre el azar
    (reglas["confidence"] >= 0.5)      # quien compra A, al menos 50% compra B
].sort_values("lift", ascending=False)

print(f"Reglas tras filtrar: {len(reglas_filtradas)}")
reglas_filtradas[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

Reglas tras filtrar: 3041


,antecedents,consequents,support,confidence,lift
116174,"frozenset({TOALLITAS BEBE PACK 3, YOGUR NATURA...","frozenset({PAÑALES T4 PACK 44, PAPILLA CEREALE...",0.050423,0.579646,6.109210
116175,"frozenset({PAÑALES T4 PACK 44, PAPILLA CEREALE...","frozenset({TOALLITAS BEBE PACK 3, YOGUR NATURA...",0.050423,0.531440,6.109210
6399,frozenset({LECHE ENTERA BRIK 1L}),"frozenset({CAFE MOLIDO 250G, LECHUGA ICEBERG UD})",0.057737,0.546448,6.054039
6398,"frozenset({CAFE MOLIDO 250G, LECHUGA ICEBERG UD})",frozenset({LECHE ENTERA BRIK 1L}),0.057737,0.639659,6.054039
6221,frozenset({LECHE ENTERA BRIK 1L}),"frozenset({CHAMPU ANTICASPA 400ML, CAFE MOLIDO...",0.060431,0.571949,6.040339
6220,"frozenset({CHAMPU ANTICASPA 400ML, CAFE MOLIDO...",frozenset({LECHE ENTERA BRIK 1L}),0.060431,0.638211,6.040339
6401,frozenset({CAFE MOLIDO 250G}),"frozenset({LECHE ENTERA BRIK 1L, LECHUGA ICEBE...",0.057737,0.506757,5.970767
6396,"frozenset({LECHE ENTERA BRIK 1L, LECHUGA ICEBE...",frozenset({CAFE MOLIDO 250G}),0.057737,0.680272,5.970767
104293,"frozenset({PAÑALES T4 PACK 44, LECHE INFANTIL ...","frozenset({PAPILLA CEREALES 600G, CHAMPU ANTIC...",0.051771,0.698701,5.922434
362,frozenset({LECHE ENTERA BRIK 1L}),frozenset({CAFE MOLIDO 250G}),0.070439,0.666667,5.851351
